In [ ]:
This is the source code used for quantum measurements in this study.

The experiment operates on a pair of qubits. First, each qubit is initialized into a superposition state using Hadamard gates. Small-angle rotation gates are then applied to introduce controlled perturbations. Next, entanglement between the two qubits is generated via CNOT operations, combined with parameterized rotation gates to create a subtle and tunable interaction structure. This configuration is designed to probe fine-grained changes in quantum state geometry while maintaining sensitivity to weak correlations.

In addition, a short pulse delay is introduced at the readout frames. This delay plays a critical role in aligning the temporal structure of the quantum measurement with external signals, particularly for correlation analysis with EEG data. The overall circuit is intentionally minimal, allowing the extraction of structural features without excessive circuit depth or noise accumulation.

It should be noted that the Rigetti Ankaa-3 device originally used in the experiment was decommissioned on April 17, 2026. Therefore, the code provided here is adapted for its successor, Cepheus-1-108Q. Full reproducibility validation on this new device remains an open task for future work.

In [ ]:
if 'experiment_counter' not in globals():
    experiment_counter = 0
experiment_counter += 1

from braket.aws import AwsDevice
from braket.circuits import Circuit
from braket.pulse import PulseSequence
from braket.device_schema.rigetti.rigetti_device_parameters_v1 import RigettiDeviceParameters

# =========================================================
# 1) DEVICE: Ankaa-3 -> Cepheus-1-108Q
# =========================================================
device = AwsDevice("arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q")

small_angle_rx = 0.05
small_angle_cry = 0.05
shots = 100
delay_seconds = 40e-6

# =========================================================
# 2) Select qubit pair
#    - First try the preferred pair (38, 39)
#    - If unavailable, automatically fall back to the first connected pair on the device
# =========================================================
preferred_pair = (38, 39)

# standardized.twoQubitProperties keys are typically in "a-b" format
two_qubit_props = getattr(device.properties.standardized, "twoQubitProperties", {})
available_edges = sorted(two_qubit_props.keys())

preferred_key = f"{preferred_pair[0]}-{preferred_pair[1]}"
preferred_key_rev = f"{preferred_pair[1]}-{preferred_pair[0]}"

if preferred_key in available_edges or preferred_key_rev in available_edges:
    q0, q1 = preferred_pair
else:
    if not available_edges:
        raise RuntimeError("No two-qubit connectivity information found on the device.")
    first_edge = available_edges[0]
    q0, q1 = map(int, first_edge.split("-"))
    print(f"Preferred pair {preferred_pair} not found. Fallback to connected pair {(q0, q1)}")

# =========================================================
# 3) CIRCUIT
# =========================================================
circuit = Circuit()
circuit.h(q0)
circuit.rx(q0, small_angle_rx)
circuit.h(q1)
circuit.cnot(q0, q1).ry(q1, small_angle_cry / 2)
circuit.cnot(q0, q1).ry(q1, -small_angle_cry / 2)

# =========================================================
# 4) Dynamically retrieve Rigetti frames from device.frames
#    Use readout_rx frames instead of hardcoding
# =========================================================
frame_0_key = f"Transmon_{q0}_readout_rx"
frame_1_key = f"Transmon_{q1}_readout_rx"

if frame_0_key not in device.frames:
    raise KeyError(f"{frame_0_key} not found in device.frames")
if frame_1_key not in device.frames:
    raise KeyError(f"{frame_1_key} not found in device.frames")

frame_0 = device.frames[frame_0_key]
frame_1 = device.frames[frame_1_key]

# =========================================================
# 5) PULSE SEQUENCE
# =========================================================
pulse_sequence = PulseSequence().delay(
    qubits_or_frames=[frame_0, frame_1],
    duration=delay_seconds
)

# Retrieve qubitCount from the actual device
qubit_count = device.properties.paradigm.qubitCount

device_parameters = RigettiDeviceParameters(
    paradigmParameters={
        "qubitCount": qubit_count,
        "pulseSequence": pulse_sequence.to_ir()
    }
)

# =========================================================
# 6) S3 Please set your S3 folder here:
# =========================================================
s3_folder = ("amazon-braket-us-west-1-xxxxxxxxxx", "tasks")

# =========================================================
# 7) RUN
# =========================================================
task = device.run(
    circuit,
    shots=shots,
    device_parameters=device_parameters,
    s3_destination_folder=s3_folder
)

result = task.result()

print(f"Experiment #{experiment_counter}")
print(f"Device ARN: {device.arn}")
print(f"Qubit pair: {(q0, q1)}")
print("Measurement Counts:", result.measurement_counts)

# Don't forget EEG recording!!!

In [ ]:
In this study, after executing the above code 30 times, the results were retrieved using the code below.

In [ ]:
import boto3
from braket.aws import AwsQuantumTask

# Create AWS Braket client
braket_client = boto3.client('braket', region_name='us-west-1')

# Retrieve the latest 30 tasks (filtered for the Rigetti Cepheus-1-108Q device)
response = braket_client.search_quantum_tasks(
    filters=[
        {'name': 'deviceArn', 'operator': 'EQUAL', 'values': ['arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q']}
    ],
    maxResults=30
)

tasks = response['quantumTasks']

# Sort by creation time
tasks_sorted = sorted(tasks, key=lambda x: x['createdAt'])

# Output with sequential numbering in the specified format
for i, task in enumerate(tasks_sorted, start=1):
    task_id = task['quantumTaskArn'].split('/')[-1]
    task_obj = AwsQuantumTask(task['quantumTaskArn'])
    result = task_obj.result()
    metadata = task_obj.metadata()

    print(f"{i}. Task ID: {task_id}")
    print(f"Measurement Counts: {result.measurement_counts}")
    print(f"Task Start: {metadata.get('createdAt', 'N/A')}")
    print(f"Task End: {metadata.get('endedAt', 'N/A')}")
    print("-" * 40)